# Data Preparation

Bagian ini merupakan langkah awal dalam *pipeline* analisis. Kita mengimpor pustaka yang dibutuhkan seperti `pandas` untuk manipulasi data, `numpy` untuk komputasi numerik, dan tentu saja modul dari `tsfel` untuk ekstraksi fitur nanti.

Pada tahap ini, kita:
1. Membaca dataset CSV dari direktori penyimpanan.
2. Memastikan kolom waktu (`time`) terformat sebagai tipe `datetime` agar dapat diurutkan secara kronologis.
3. Melakukan pembersihan tipe data awal. Sangat penting untuk memaksa kolom target (`NO2`) menjadi tipe data numerik. Terkadang data dari sensor mengandung teks atau *error code*. Kode `pd.to_numeric(..., errors='coerce')` akan menyapu bersih nilai-nilai teks tersebut menjadi data kosong (`NaN`). Kita juga mencetak berapa banyak data awal yang terdeteksi sebagai bukan angka.



In [1]:
import pandas as pd
import numpy as np
import inspect
import tsfel.feature_extraction.features as tsfel_features

# ---------- 1. Muat dan bersihkan data ----------
# Membaca data polutan NO2
df = pd.read_csv('polutan_Manyar_2025_2026.csv')
print(df.columns) # <--- Tambahkan baris ini untuk mengecek
df['time'] = pd.to_datetime(df['time'])
df = df.sort_values('time').reset_index(drop=True)

target_pollutant = 'NO2'

# Memastikan kolom target bertipe numerik, error menjadi NaN
df[target_pollutant] = pd.to_numeric(df[target_pollutant], errors='coerce')

n_missing_before = df[target_pollutant].isna().sum()
print(f"Jumlah nilai non-numerik/kosong awal yang dikonversi jadi NaN: {n_missing_before}")


Index(['time', 'NO2', 'CO', 'O3'], dtype='object')
Jumlah nilai non-numerik/kosong awal yang dikonversi jadi NaN: 177


# Tahap 2: Deteksi Outliers (Pencilan) dengan Metode IQR

Data sensor seringkali menangkap lonjakan polusi yang tidak wajar akibat gangguan alat (*noise*) atau kesalahan pencatatan. Untuk mengatasinya, kita menggunakan pendekatan statistik **Interquartile Range (IQR)**.

Prosesnya adalah:
1. Menemukan rentang mayoritas data terpusat (antara Kuartil 1 dan Kuartil 3).
2. Menghitung rentang IQR.
3. Membuat 'pagar pembatas'. Nilai wajar diasumsikan berada dalam rentang `[Q1 - 1.5*IQR]` hingga `[Q3 + 1.5*IQR]`.
4. Jika ada data NO2 yang melampaui batas atas atau merosot di bawah batas bawah, data tersebut dicoret dan statusnya diganti menjadi `NaN` agar nantinya bisa diisi ulang (diimputasi) dengan nilai yang lebih logis.



In [2]:
# ---------- 2. Deteksi dan Penghapusan Outliers (Pencilan) ----------
# Menghitung Kuartil 1 (Q1) dan Kuartil 3 (Q3)
Q1 = df[target_pollutant].quantile(0.25)
Q3 = df[target_pollutant].quantile(0.75)

# Menghitung Interquartile Range (IQR)
IQR = Q3 - Q1

# Menentukan batas kewajaran data
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Batas Bawah IQR: {lower_bound:.2f} | Batas Atas IQR: {upper_bound:.2f}")

# Menghapus nilai yang melanggar batas (diubah menjadi NaN)
outliers_condition = (df[target_pollutant] < lower_bound) | (df[target_pollutant] > upper_bound)
df.loc[outliers_condition, target_pollutant] = np.nan

print(f"Jumlah outliers yang terdeteksi dan dikosongkan: {outliers_condition.sum()}")


Batas Bawah IQR: -0.00 | Batas Atas IQR: 0.00
Jumlah outliers yang terdeteksi dan dikosongkan: 9


# Tahap 3: Imputasi Missing Value

Setelah tahap pertama dan kedua, data kita mungkin memiliki banyak "lubang" (nilai `NaN`). Lubang ini berasal dari konversi paksa teks, data yang memang kosong dari sensor, maupun nilai *outlier* yang baru saja kita hapus. Algoritma ekstraksi fitur deret waktu membutuhkan data yang saling bersambung sempurna tanpa jeda kosong.

Oleh karena itu, kita melakukan imputasi menggunakan **Interpolasi Waktu** (`interpolate(method='time')`). Metode ini akan memperkirakan nilai yang hilang dengan menarik garis tren antara data sebelum dan sesudah lubang tersebut, dengan memperhitungkan jarak waktunya. 

Sebagai pengaman terakhir, kita menerapkan `ffill()` (mengambil nilai terakhir untuk menutupi bagian kosong setelahnya) dan `bfill()` (mengambil nilai sesudahnya untuk menutupi kosong sebelumnya) demi memastikan data $100\%$ terisi tanpa sisa.



In [3]:
# ---------- 3. Imputasi Missing Value ----------
# Jadikan kolom tanggal sebagai index sementara untuk interpolasi
df_clean = df.set_index('time')

# Melakukan interpolasi berbasis waktu
df_clean = df_clean.interpolate(method='time')

# Menambal celah di awal atau akhir data jika interpolasi tidak menjangkau
df_clean = df_clean.ffill().bfill()

print(f"Sisa missing value setelah proses imputasi: {df_clean[target_pollutant].isna().sum()}")


Sisa missing value setelah proses imputasi: 0


# Tahap 4: Persiapan Data untuk TSFEL

Data yang sudah bersih kini diekstrak dari DataFrame *pandas* menjadi format *array* 1 dimensi (`signal_1d`) menggunakan `numpy`. Format *array* inilah yang dapat dipahami dan diproses secara matematis oleh pustaka TSFEL.

Kita juga mendefinisikan variabel `fs = 1` (*sampling frequency*) serta menyiapkan *string* panjang berisi daftar persis 68 fitur yang ditargetkan. Fitur-fitur ini sangat bervariasi, meliputi domain statistik dasar (seperti `calc_mean`, `kurtosis`), domain temporal (seperti `zero_cross`, `autocorr`), hingga domain spektral dan fraktal (seperti `spectral_entropy`, `higuchi_fractal_dimension`). Daftar *string* tersebut kemudian dipecah (`.split()`) menjadi sebuah struktur *list* Python.



In [4]:
fs = 1
signal_1d = df_clean[target_pollutant].astype(float).values

# Daftar PERSIS fitur (68 Fitur)
FEATURE_LIST = """abs_energy auc autocorr average_power calc_centroid calc_max calc_mean
calc_median calc_min calc_std calc_var dfa distance ecdf ecdf_percentile ecdf_percentile_count
ecdf_slope entropy fundamental_frequency higuchi_fractal_dimension hist_mode human_range_energy
hurst_exponent interq_range kurtosis lempel_ziv lpcc max_frequency max_power_spectrum
maximum_fractal_length mean_abs_deviation mean_abs_diff mean_diff median_abs_deviation
median_abs_diff median_diff median_frequency mfcc mse negative_turning neighbourhood_peaks
petrosian_fractal_dimension pk_pk_distance positive_turning power_bandwidth rms skewness slope
spectral_centroid spectral_decrease spectral_distance spectral_entropy spectral_kurtosis
spectral_positive_turning spectral_roll_off spectral_roll_on spectral_skewness spectral_slope
spectral_spread spectral_variation spectrogram_mean_coeff sum_abs_diff wavelet_abs_mean
wavelet_energy wavelet_entropy wavelet_std wavelet_var zero_cross""".split()

print("Jumlah fitur yang disiapkan untuk diekstrak:", len(FEATURE_LIST))


Jumlah fitur yang disiapkan untuk diekstrak: 68


# Tahap 5: Eksekusi TSFEL dan Penyimpanan Hasil

Tahap terakhir adalah mengeksekusi ekstraksi secara dinamis. Pustaka TSFEL memiliki ragam *output*; beberapa fungsi mengeluarkan angka tunggal (skalar), sementara yang lain mungkin mengeluarkan *array*. 

1. **`to_scalar`**: Fungsi pembantu ini bertugas menyeragamkan bentuk. Apapun yang dikeluarkan oleh TSFEL, jika berupa *array*, ia akan dipadatkan menjadi metrik tunggal menggunakan nilai rata-rata (`np.nanmean`), sehingga format data tetap konsisten untuk tabel *machine learning*.
2. **`extract_one`**: Fungsi pembantu ini memanfaatkan metode inspeksi bawaan Python. Ia mencocokkan nama fitur dalam *list* kita dengan modul pustaka `tsfel_features`. Ia juga mengecek apakah fungsi tersebut memerlukan parameter waktu (`fs`) atau tidak sebelum menjalankannya.
3. Terakhir, *loop* akan berjalan melintasi ke-68 fitur. Semua hasilnya dikumpulkan menjadi sebuah kamus (*dictionary*) yang kemudian dibungkus menjadi sebuah baris tunggal dalam Pandas DataFrame. Baris pamungkas ini kemudian di-ekspor dalam format `.csv`.



In [10]:
# ---------- 5. Proses Ekstraksi Fitur ----------
def to_scalar(result):
    # Mengubah hasil dictionary dari TSFEL menjadi nilai mentah
    if isinstance(result, dict) and "values" in result:
        result = result["values"]
    # Jika hasil berupa array atau list panjang, kita ambil rata-ratanya
    if isinstance(result, (list, tuple, np.ndarray)):
        arr = np.asarray(result, dtype=float)
        return float(np.nanmean(arr))
    # Jika sudah skalar, ubah jadi float standar
    return float(result)

def extract_one(fn_name, signal, fs):
    # Memanggil fungsi TSFEL secara dinamis berdasarkan nama fiturnya
    fn = getattr(tsfel_features, fn_name)
    params = inspect.signature(fn).parameters
    # Memeriksa apakah fungsi tersebut butuh parameter 'fs'
    if "fs" in params:
        result = fn(signal, fs)
    else:
        result = fn(signal)
    return to_scalar(result)

# Menjalankan iterasi untuk setiap fitur pada sinyal NO2
row = {}
for fn_name in FEATURE_LIST:
    row[fn_name] = extract_one(fn_name, signal_1d, fs)

extracted_features_final = pd.DataFrame([row])

print(f"Berhasil! Ekstraksi menghasilkan {extracted_features_final.shape[1]} fitur.")

# Menyimpan hasil ke dalam format CSV
output_filename = f'{target_pollutant}_Manyar_TSFEL.csv'
extracted_features_final.to_csv(output_filename, index=False)
print(f"File berhasil disimpan sebagai: {output_filename}")


Berhasil! Ekstraksi menghasilkan 68 fitur.
File berhasil disimpan sebagai: NO2_Manyar_TSFEL.csv
